In [4]:
import torch
from torch.utils.data import Dataset
import torchvision.io as io
import torchvision.transforms.functional as F
from transformers import BertTokenizer

class TikTokDataset(Dataset):
    def __init__(self, csv_file, video_dir, batch_size, transform=None):
        self.data = csv_file
        self.video_dir = video_dir
        self.transform = transform
        self.num_important_frames = 8  # 4 fixes + 4 pics de mouvement
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.y_train = self.data['popularity']  

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 1. Récupérer le chemin de la vidéo
        video_path = self.get_path(idx) 
        
        # 2. Charger la vidéo (on récupère les frames)
        # vframes est un tenseur [T, H, W, C]
        vframes, _, _ = io.read_video(video_path, pts_unit='sec')
        
        # 3. Calculer le mouvement entre les frames
        diff = torch.abs(torch.diff(vframes, dim=0))  # [T-1, H, W, C]
        motion_scores = diff.mean(dim=[1, 2, 3])  # [T-1]

        # 4. Sélectionner tes 8 indices (4 fixes + 4 pics de mouvement)
        fixed_times = torch.linspace(0, len(motion_scores)-1, steps=self.num_important_frames//2+1, dtype=torch.int)  # 4 indices fixes
        selected_frames = vframes[fixed_times]  # [4, H, W, C]
        for i in range(self.num_important_frames// 2):
            segment = motion_scores[fixed_times[i]:fixed_times[i+1]]
            peak_idx = torch.argmax(segment) + fixed_times[i]  # index du pic de mouvement
            selected_frames = torch.cat((selected_frames, vframes[peak_idx].unsqueeze(0)), dim=0)  # ajouter le pic
        
        # 5. Redimensionnement et normalisation
        selected_frames = selected_frames.to(torch.float32) / 255.0
        selected_frames = F.resize(selected_frames, [224, 224])
        selected_frames = torch.permute(selected_frames, 0, 3, 1, 2)  # [num_frames, C, H, W]
        
        # 6.. Préparation du texte (avec un tokenizer BERT)
        description = self.data.iloc[idx]['description']
        text_inputs = self.tokenizer(description, padding='max_length', max_length=50, truncation=True, return_tensors="pt")

        # 7. Préparation du tabulaire
        # On sélectionne les colonnes numériques
        tabular_data = torch.tensor(self.data.iloc[idx][['video_duration', 'aspect_ratio']].values, dtype=torch.float)

        # 8. La cible (Y)
        label = torch.tensor(self.y_train.iloc[idx], dtype=torch.float)

        return selected_frames, text_inputs['input_ids'].squeeze(), tabular_data, label